# Notebook 04 — Product Category Clustering
## NLP Business Case: Automated Customer Reviews
### Ironhack Bootcamp Project

---

In this notebook, I apply **unsupervised clustering** to discover 4–6 meta-categories of products
hidden in Amazon review texts. Unlike the supervised sentiment classifiers in Notebooks 02 and 03,
this task has **no pre-existing labels** — the algorithm must learn the category structure from the text alone.

**Pipeline**:  Review Text → MiniLM Embeddings (384-dim) → MiniBatchKMeans → UMAP Visualisation

### Models Used

| Component          | Model                      | Why                                                          |
|--------------------|----------------------------|--------------------------------------------------------------|
| **Embedding**      | `all-MiniLM-L6-v2`         | ~22 MB, 384 dimensions, very fast — ideal for clustering     |
| **Clustering**     | `MiniBatchKMeans`          | Processes data in random batches — handles large volumes     |
| **Visualisation**  | `UMAP`                     | Preserves local + global structure better than t-SNE         |

### Metrics
- **Silhouette Score** — measures how well-separated the clusters are (range -1 to 1)
- **Elbow Method (Inertia)** — helps select the optimal number of clusters `k`
- **Top Terms per Cluster** — TF-IDF weighted keywords that characterise each cluster

**Data source**: Arrow-format test split produced by Notebook 01, merged with sentiment
predictions from Notebooks 02 and 03.

---

## Section 0 — Setup & Environment

In [ ]:
# ── 0.1  Detect execution environment ────────────────────────────────────────
# Check whether we are running inside Google Colab or locally.
# This flag controls whether we mount Google Drive and install packages.
try:
    import google.colab          # noqa: F401  (import only for detection)
    IN_COLAB = True              # Running on Colab
except ImportError:
    IN_COLAB = False             # Running locally

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# ── 0.2  Mount Google Drive (Colab only) ─────────────────────────────────────
# Google Drive is mounted so that all artifacts survive runtime resets.
# The mount point /content/drive is the standard Colab convention.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")   # Prompts for Google account auth
    print("Google Drive mounted at /content/drive")
else:
    print("Skipping Drive mount — running locally.")

In [ ]:
# ── 0.3  Install required packages ───────────────────────────────────────────
# Install all clustering and NLP dependencies.
# 'sentence-transformers' provides the MiniLM embedding model.
# 'umap-learn' is the dimensionality reduction library for visualisation.
if IN_COLAB:
    import subprocess
    subprocess.run([
        "pip", "install", "-q", "--upgrade",
        "sentence-transformers",  # MiniLM and other embedding models
        "umap-learn",             # UMAP dimensionality reduction
        "datasets",               # HuggingFace Datasets (Arrow format loading)
        "scikit-learn",           # MiniBatchKMeans, Silhouette Score, etc.
        "pandas",                 # DataFrame operations
        "numpy",                  # Numerical ops and array storage
        "matplotlib",             # Plotting (scatter, bar charts)
        "seaborn",                # Statistical data visualisation
        "tqdm",                   # Progress bars
    ], check=True)
    print("All packages installed.")
else:
    print("Skipping pip install — manage dependencies locally via requirements.txt.")

In [ ]:
# ── 0.4  Standard library and third-party imports ────────────────────────────
import os                          # OS path utilities
import json                        # JSON I/O for saving metrics
import random                      # Python random seed
import warnings                    # Suppress non-critical warnings
warnings.filterwarnings("ignore")  # Keep output clean

import numpy as np                 # Numerical operations
import pandas as pd                # DataFrames for predictions and cluster labels
import matplotlib.pyplot as plt    # Plotting (scatter, bar charts)
import seaborn as sns              # Statistical data visualisation

from datasets import load_from_disk  # HuggingFace Datasets — load Arrow format

from sentence_transformers import SentenceTransformer  # MiniLM embedding model

from sklearn.cluster import MiniBatchKMeans    # Scalable K-Means for large data
from sklearn.metrics import silhouette_score   # Cluster quality metric
from sklearn.feature_extraction.text import TfidfVectorizer  # Top terms per cluster

from umap import UMAP              # Dimensionality reduction for visualisation

from tqdm.auto import tqdm         # Auto-selects notebook or terminal progress bar

print("All imports successful.")

In [ ]:
# ── 0.5  Set random seeds for full reproducibility ───────────────────────────
# Fix seeds in Python, NumPy so results are deterministic across runs.
# MiniBatchKMeans and UMAP also accept a random_state parameter (set later).
SEED = 42

random.seed(SEED)                          # Python built-in random module seed
np.random.seed(SEED)                       # NumPy global seed

print(f"Random seeds fixed to {SEED}.")

In [ ]:
# ── 0.6  Define all project paths ────────────────────────────────────────────
# All paths are centralised here so changing them once propagates everywhere.
# BASE_DIR   : root of the project on Google Drive (or local equivalent)
# DATASET_DIR : Arrow dataset saved by NB01 (input)
# OUTPUT_DIR  : RoBERTa CSV (input) + cluster CSV/JSON (output) — shared with N05
# PLOTS_DIR   : PNG files (UMAP scatter, silhouette bar, elbow curve)

if IN_COLAB:
    # Google Drive path — adjust the folder name if yours differs
    BASE_DIR = "/content/drive/MyDrive/nlp-project/business-case-01"
else:
    # Local development path — update to your local project root
    BASE_DIR = os.path.expanduser("~/+Dev/nlp-businesscase")

DATASET_DIR = os.path.join(BASE_DIR, "data", "dataset")   # Arrow dataset from NB01
OUTPUT_DIR  = os.path.join(BASE_DIR, "data")              # RoBERTa CSV + cluster outputs (shared with N05)
PLOTS_DIR   = os.path.join(BASE_DIR, "data", "plots")     # PNG visualisations
MODELS_DIR  = os.path.join(BASE_DIR, "data", "models")    # Saved embeddings

# Create directories if they do not yet exist
os.makedirs(PLOTS_DIR,  exist_ok=True)   # No error if directory already exists
os.makedirs(MODELS_DIR, exist_ok=True)   # Same for models directory
os.makedirs(OUTPUT_DIR,  exist_ok=True)   # Same for output data directory

print(f"BASE_DIR    : {BASE_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"PLOTS_DIR   : {PLOTS_DIR}")
print(f"MODELS_DIR  : {MODELS_DIR}")

---
## Section 1 — Data Loading

I load the Arrow-format test split produced by Notebook 01. I use the **test split** for clustering
because:
1. It is the smallest split (15% of data) — manageable for MiniLM encoding and UMAP visualisation
2. Sentiment predictions from NB02 and NB03 were generated on this split — I can overlay sentiment
   colours on clusters to analyse sentiment distribution per category

I also load the sentiment prediction CSVs from Notebooks 02 and 03 to enrich the cluster analysis.
If only one prediction CSV is available, I work with whichever is present.

In [ ]:
# ── 1.1  Load Arrow dataset from Notebook 01 ─────────────────────────────────
# NB01 saved a DatasetDict with keys 'train', 'validation', 'test'.
# I only load the test split — it is the right size for clustering.
#
# Expected output: DatasetDict with 3 splits, ~15% of total data in test.

if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(
        f"Arrow dataset not found at {DATASET_DIR}. "
        "Run Notebook 01 first to generate the preprocessed dataset."
    )

print(f"Loading Arrow dataset from: {DATASET_DIR}")

dataset = load_from_disk(DATASET_DIR)  # Reloads the DatasetDict saved by NB01

print(f"Dataset loaded:")
print(dataset)  # Shows split names and row counts

# Extract the test split as a DataFrame for easier manipulation
df_test = dataset["test"].to_pandas()  # Convert Arrow → pandas DataFrame

print(f"\nTest split DataFrame shape: {df_test.shape}")
print(f"Columns: {list(df_test.columns)}")
print(f"\nFirst 3 rows:")
display(df_test.head(3))

In [ ]:
# ── 1.2  Load sentiment predictions from Notebooks 02 and 03 ─────────────────
# Each prediction CSV contains: text, true_label, predicted_label, confidence.
# I try to load both — if one is missing (e.g., only one model was run),
# I continue with whichever is available.
#
# Expected output: one or two prediction DataFrames loaded.

predictions = {}  # Dictionary to store available prediction DataFrames

# ── Try loading DistilBERT predictions (NB02) ──
distilbert_csv = os.path.join(DATASET_DIR, "predictions_distilbert.csv")
if os.path.exists(distilbert_csv):
    preds_dbert = pd.read_csv(distilbert_csv, encoding="utf-8")
    predictions["distilbert"] = preds_dbert
    print(f"✓ Loaded DistilBERT predictions : {len(preds_dbert):,} rows")
else:
    print(f"✗ DistilBERT predictions not found at: {distilbert_csv}")

# ── Try loading RoBERTa predictions (NB03) ──
# N03 saves to data/predictions_roberta.csv — matches OUTPUT_DIR
roberta_csv = os.path.join(OUTPUT_DIR, "predictions_roberta.csv")
if os.path.exists(roberta_csv):
    preds_roberta = pd.read_csv(roberta_csv, encoding="utf-8")
    predictions["roberta"] = preds_roberta
    print(f"✓ Loaded RoBERTa predictions     : {len(preds_roberta):,} rows")
else:
    print(f"✗ RoBERTa predictions not found at: {roberta_csv}")

# ── Select primary prediction source for analysis ──
# Prefer RoBERTa (better model) if available, otherwise fall back to DistilBERT.
PRED_SOURCE = None
if "roberta" in predictions:
    df_preds = predictions["roberta"].copy()
    PRED_SOURCE = "roberta"
elif "distilbert" in predictions:
    df_preds = predictions["distilbert"].copy()
    PRED_SOURCE = "distilbert"
else:
    df_preds = None
    print("\n⚠️  No sentiment predictions available. Cluster analysis will use true labels from NB01.")

if df_preds is not None:
    print(f"\nUsing '{PRED_SOURCE}' predictions for sentiment enrichment.")
    print(f"Prediction columns: {list(df_preds.columns)}")

In [ ]:
# ── 1.3  Merge test data with sentiment predictions ──────────────────────────
# I merge on the 'text' column to attach predicted labels to each review.
# This enriches the cluster analysis with sentiment information.
#
# The test DataFrame from NB01 has: text, label (true sentiment), rating, category.
# The predictions from NB02/NB03 have: text, true_label, predicted_label, confidence.
#
# Expected output: merged DataFrame with both true and predicted labels.

if df_preds is not None:
    # Merge on 'text' — keep NB01 columns, add predicted_label and confidence
    df_merged = df_test.merge(
        df_preds[["text", "predicted_label", "confidence"]],
        on="text",
        how="left"  # Keep all test rows even if prediction is missing
    )
    print(f"Merged DataFrame shape: {df_merged.shape}")
    print(f"Columns: {list(df_merged.columns)}")
else:
    # No predictions available — use the test DataFrame as-is
    df_merged = df_test.copy()
    print("No predictions to merge — using test data as-is.")
    print(f"DataFrame shape: {df_merged.shape}")

print(f"\nMissing values per column:")
print(df_merged.isnull().sum())

In [ ]:
# ── 1.4  Sampling strategy for large datasets ─────────────────────────────────
# If the test set is very large (> 50 000 reviews), I take a stratified sample
# to keep MiniLM encoding and UMAP visualisation fast and memory-safe.
# Stratification by 'label' (true sentiment) preserves the class distribution.
#
# Expected output: Sample size printed; original shape unchanged if under threshold.

MAX_SAMPLES = 50_000  # Maximum reviews to process — tune based on your RAM/GPU

if len(df_merged) > MAX_SAMPLES:
    print(f"Test set has {len(df_merged):,} reviews — sampling {MAX_SAMPLES:,}...")

    # Stratified sample by true label to preserve class proportions
    df_merged = df_merged.groupby("label", group_keys=False).apply(
        lambda g: g.sample(
            n=min(len(g), int(MAX_SAMPLES * len(g) / len(df_merged))),
            random_state=SEED
        )
    ).reset_index(drop=True)

    print(f"  → Sampled down to {len(df_merged):,} reviews.")
else:
    print(f"Test set has {len(df_merged):,} reviews — no sampling needed (≤ {MAX_SAMPLES:,}).")

# Show final class distribution
print(f"\nClass distribution after sampling:")
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
for lbl, count in df_merged["label"].value_counts().sort_index().items():
    pct = 100 * count / len(df_merged)
    print(f"  {label_names[lbl]:10s} ({lbl}): {count:>8,}  ({pct:5.1f}%)")

---
## Section 2 — Generating Text Embeddings

I convert each review text into a dense 384-dimensional vector using `all-MiniLM-L6-v2`.
This model from Sentence Transformers is optimised for semantic similarity — texts with similar
meanings get vectors that are close together in embedding space, which is exactly what clustering needs.

**Why MiniLM and not a larger model?**
- Only **22 MB** — downloads in seconds even on Colab free tier
- **384 dimensions** — compact enough for MiniBatchKMeans to be fast
- Trained on **1B+ sentence pairs** — captures general semantic meaning well
- **No GPU required** — runs efficiently on CPU, saving GPU quota for NB02/NB03 training

**Reference**: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [ ]:
# ── 2.1  Load the MiniLM embedding model ──────────────────────────────────────
# SentenceTransformer downloads the model from HuggingFace Hub on first call.
# The model is cached locally for subsequent runs (no re-download needed).
#
# Expected output: model loaded successfully, embedding dimension printed.

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  # Sentence Transformers model identifier

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Get the embedding dimension from a test encoding
test_embedding = embedding_model.encode(["test sentence"], show_progress_bar=False)
EMBEDDING_DIM = test_embedding.shape[1]

print(f"✓ Model loaded. Embedding dimension: {EMBEDDING_DIM}")
print(f"  Model max sequence length: {embedding_model.max_seq_length} tokens")

In [ ]:
# ── 2.2  Encode all review texts to embeddings ───────────────────────────────
# .encode() converts a list of strings to a NumPy array of shape (n_reviews, 384).
# show_progress_bar=True displays a tqdm bar — useful for large datasets.
# normalize_embeddings=True projects vectors onto the unit sphere, which improves
# cosine-similarity-based clustering (MiniBatchKMeans uses Euclidean distance,
# but normalised vectors + Euclidean ≈ cosine similarity).
#
# Expected output: NumPy array of shape (n_reviews, 384).

review_texts = df_merged["text"].tolist()  # List of all review strings

print(f"Encoding {len(review_texts):,} reviews ...")
print(f"  Model       : {EMBEDDING_MODEL_NAME}")
print(f"  Output dim  : {EMBEDDING_DIM}")
print(f"  Normalise   : True (projected to unit sphere)")

embeddings = embedding_model.encode(
    review_texts,
    show_progress_bar=True,         # tqdm progress bar
    normalize_embeddings=True,      # L2-normalise → better for clustering
    batch_size=64,                  # Batch size for encoding (tune for your RAM)
)

print(f"\n✓ Embeddings generated.")
print(f"  Shape       : {embeddings.shape}")
print(f"  Dtype       : {embeddings.dtype}")
print(f"  Memory      : {embeddings.nbytes / 1e6:.2f} MB")
print(f"  Norm check  : mean L2 norm = {np.linalg.norm(embeddings, axis=1).mean():.6f} (should be ~1.0)")

In [ ]:
# ── 2.3  Save embeddings to disk ─────────────────────────────────────────────
# I save the embeddings as a compressed NumPy file so NB05 can reuse them
# without re-encoding. .npz format compresses well for float32 arrays.
#
# Expected output: .npz file saved to OUTPUT_DIR.

embeddings_path = os.path.join(OUTPUT_DIR, "embeddings_minilm.npz")

np.savez_compressed(embeddings_path, embeddings=embeddings)

file_size_mb = os.path.getsize(embeddings_path) / 1e6
print(f"✓ Embeddings saved to: {embeddings_path}")
print(f"  File size: {file_size_mb:.2f} MB")

In [ ]:
# ── 2.4  Verify embedding quality — pairwise similarity check ────────────────
# I compute cosine similarity between a few example pairs to sanity-check
# that similar reviews have higher similarity than dissimilar ones.
# Cosine similarity on normalised vectors is simply the dot product.
#
# Expected output: similar reviews → high similarity (> 0.5); different → low (< 0.3).

# Pick two reviews from the same sentiment class (should be similar)
same_class_mask = df_merged["label"] == df_merged["label"].mode()[0]
same_class_indices = df_merged[same_class_mask].index[:2]
sim_review_a = df_merged.loc[same_class_indices[0], "text"][:100]
sim_review_b = df_merged.loc[same_class_indices[1], "text"][:100]
sim_score = np.dot(embeddings[same_class_indices[0]], embeddings[same_class_indices[1]])

# Pick two reviews from different sentiment classes (should be dissimilar)
neg_idx = df_merged[df_merged["label"] == 0].index[0]
pos_idx = df_merged[df_merged["label"] == 2].index[0]
diff_review_a = df_merged.loc[neg_idx, "text"][:100]
diff_review_b = df_merged.loc[pos_idx, "text"][:100]
diff_score = np.dot(embeddings[neg_idx], embeddings[pos_idx])

print("Embedding quality check — cosine similarity (normalised → dot product):")
print(f"\n  Similar-class reviews:")
print(f"    A: \"{sim_review_a}...\"")
print(f"    B: \"{sim_review_b}...\"")
print(f"    → Similarity: {sim_score:.4f}  (higher = more similar)")
print(f"\n  Different-class reviews:")
print(f"    A (Neg): \"{diff_review_a}...\"")
print(f"    B (Pos): \"{diff_review_b}...\"")
print(f"    → Similarity: {diff_score:.4f}  (should be lower)")

if sim_score > diff_score:
    print("\n✅ Embeddings look good — similar reviews are closer in vector space.")
else:
    print("\n⚠️  Unexpected: dissimilar reviews are closer. Investigate.")

---
## Section 3 — Clustering with MiniBatchKMeans

Now the core unsupervised learning step: grouping reviews into product categories.

**Why MiniBatchKMeans instead of regular K-Means?**
- Regular K-Means loads the **entire dataset** into memory at once — O(n·k·d) per iteration
- MiniBatchKMeans processes **random mini-batches** — memory usage is O(batch_size·k·d), regardless of dataset size
- This makes it viable for 571M reviews on Colab's limited RAM (12 GB free tier)

**How I choose k (number of clusters)**:
1. **Elbow Method** — plot inertia (within-cluster sum of squares) for k=2…10;
   the "elbow" where the curve bends indicates a good k.
2. **Silhouette Score** — measures how similar each point is to its own cluster vs. neighbouring clusters.
   Higher is better (range -1 to 1).
3. **Domain reasoning** — Amazon has ~33 categories; I aim to discover 4–6 meta-categories.

In [ ]:
# ── 3.1  Elbow Method & Silhouette Score for k=2…10 ─────────────────────────
# I run MiniBatchKMeans for multiple values of k and collect:
#   - Inertia (within-cluster sum of squares) → elbow curve
#   - Silhouette Score → how well-separated the clusters are
#
# MiniBatchKMeans is much faster than KMeans for this sweep.
# Expected output: two line plots (elbow + silhouette) and a table of results.

K_VALUES  = range(2, 11)          # Try k from 2 to 10 clusters
BATCH_SIZE = 1024                 # Mini-batch size for KMeans iterations
N_INIT     = 10                    # Number of random initialisations (best picked)

inertias = []                     # Within-cluster sum of squares per k
silhouettes = []                  # Silhouette score per k (computed on a sample)

# For silhouette score on large datasets, use a random sample to save time
SIL_SAMPLE_SIZE = min(10_000, len(embeddings))  # Cap at 10k for speed
rng = np.random.default_rng(SEED)  # Explicit RNG for reproducibility
sil_sample_indices = rng.choice(len(embeddings), size=SIL_SAMPLE_SIZE, replace=False)
sil_sample = embeddings[sil_sample_indices]

print(f"Sweeping k = {min(K_VALUES)} … {max(K_VALUES)} clusters ...")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Silhouette sample: {SIL_SAMPLE_SIZE:,} points")
print()

for k in tqdm(K_VALUES, desc="Sweeping k"):
    # Fit MiniBatchKMeans
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        batch_size=BATCH_SIZE,
        n_init=N_INIT,
        random_state=SEED,
        max_iter=100,
        verbose=0,
    )
    kmeans.fit(embeddings)

    # Collect inertia
    inertias.append(kmeans.inertia_)

    # Predict cluster labels for the silhouette sample
    sil_labels = kmeans.predict(sil_sample)

    # Compute Silhouette Score on the sample
    sil = silhouette_score(sil_sample, sil_labels)
    silhouettes.append(sil)

    print(f"  k={k:2d}  |  Inertia={inertias[-1]:>12,.0f}  |  Silhouette={sil:.4f}")

print("\n✓ K-sweep complete.")

In [ ]:
# ── 3.2  Plot Elbow Curve and Silhouette Score ──────────────────────────────
# Two side-by-side plots to guide the choice of k.
# - Left: Elbow curve — look for the "bend" where inertia stops dropping fast.
# - Right: Silhouette — higher is better; a sharp drop suggests over-clustering.
#
# Expected output: two-plot figure saved as PNG.

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Elbow curve ──
ax1.plot(K_VALUES, inertias, marker="o", color="#0891B2", linewidth=2, markersize=8)
ax1.set_title("Elbow Method — Inertia vs. k", fontsize=13, fontweight="bold")
ax1.set_xlabel("Number of Clusters (k)", fontsize=11)
ax1.set_ylabel("Inertia (within-cluster SSE)", fontsize=11)
ax1.grid(True, linestyle="--", alpha=0.5)
# Highlight the range 4-6 with a shaded region (our target)
ax1.axvspan(4, 6, alpha=0.1, color="#334155", label="Target range (4–6)")
ax1.legend(fontsize=10)

# ── Silhouette curve ──
ax2.plot(K_VALUES, silhouettes, marker="s", color="#DC2626", linewidth=2, markersize=8)
ax2.set_title("Silhouette Score vs. k", fontsize=13, fontweight="bold")
ax2.set_xlabel("Number of Clusters (k)", fontsize=11)
ax2.set_ylabel("Silhouette Score", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.axvspan(4, 6, alpha=0.1, color="#334155", label="Target range (4–6)")
ax2.legend(fontsize=10)

fig.suptitle("MiniBatchKMeans — Optimal k Selection", fontsize=14, fontweight="bold")
plt.tight_layout()

k_select_path = os.path.join(PLOTS_DIR, "nb04_k_selection.png")
plt.savefig(k_select_path, dpi=150, bbox_inches="tight")
print(f"✓ K-selection plot saved → {k_select_path}")
plt.show()

In [ ]:
# ── 3.3  Select best k and fit final model ───────────────────────────────────
# I pick the k with the highest silhouette score within the 4–6 target range.
# This balances statistical quality with the project requirement of 4–6 meta-categories.
#
# Expected output: best k printed, final MiniBatchKMeans fitted.

# ── Find best k in the 4–6 range ──
target_k_values = [k for k in K_VALUES if 4 <= k <= 6]
target_silhouettes = [silhouettes[list(K_VALUES).index(k)] for k in target_k_values]

best_k_idx = np.argmax(target_silhouettes)     # Index within target range
BEST_K = target_k_values[best_k_idx]           # The actual k value
BEST_SIL = target_silhouettes[best_k_idx]      # Its silhouette score

print(f"Best k in range 4–6: {BEST_K}  (Silhouette = {BEST_SIL:.4f})")

# Show full results table for reference
print(f"\nFull k-sweep results:")
print(f"  {'k':>3s}  |  {'Inertia':>12s}  |  {'Silhouette':>10s}")
print(f"  {'-'*32}")
for k, inert, sil in zip(K_VALUES, inertias, silhouettes):
    marker = " ← BEST" if k == BEST_K else ""
    print(f"  {k:3d}  |  {inert:>12,.0f}  |  {sil:>10.4f}{marker}")

In [ ]:
# ── 3.4  Fit final KMeans model with best k ─────────────────────────────────
# I re-fit with the selected k on the full embedding set.
# n_init is increased to 10 for a more thorough initialisation search.
#
# Expected output: final model fitted, cluster labels assigned.

print(f"Fitting final MiniBatchKMeans with k={BEST_K} ...")

final_kmeans = MiniBatchKMeans(
    n_clusters=BEST_K,
    batch_size=BATCH_SIZE,
    n_init=10,                     # More initialisations for better convergence
    random_state=SEED,
    max_iter=200,                  # More iterations for final fit
    verbose=1,                     # Show progress during fitting
)
final_kmeans.fit(embeddings)

# ── Assign cluster labels to all reviews ──
cluster_labels = final_kmeans.labels_  # Integer labels: 0 … BEST_K-1

# Attach cluster labels to the DataFrame
df_merged["cluster"] = cluster_labels

print(f"\n✓ Final clustering complete.")

# ── Recompute Silhouette on the final model ──
# The k-sweep used n_init=3; the final model uses n_init=10 and may
# converge differently. I recompute the silhouette here for accurate reporting.
final_sil_labels = final_kmeans.predict(sil_sample)
FINAL_SIL = silhouette_score(sil_sample, final_sil_labels)
print(f"  Final Silhouette Score: {FINAL_SIL:.4f}  (recomputed on final model)")
print(f"  Final inertia: {final_kmeans.inertia_:,.0f}")
print(f"  Iterations   : {final_kmeans.n_iter_}")
print(f"  Clusters     : {BEST_K}")

In [ ]:
# ── 3.5  Cluster size distribution ───────────────────────────────────────────
# A healthy clustering has clusters of comparable size — one giant cluster
# and many tiny ones suggests the embedding didn't capture enough structure.
#
# Expected output: bar chart of cluster sizes, printed counts.

cluster_counts = df_merged["cluster"].value_counts().sort_index()

print("Cluster size distribution:")
print(f"  {'Cluster':>8s}  |  {'Count':>8s}  |  {'%':>6s}")
print(f"  {'-'*32}")
for cluster_id in sorted(cluster_counts.index):
    count = cluster_counts[cluster_id]
    pct = 100 * count / len(df_merged)
    print(f"  {cluster_id:8d}  |  {count:>8,}  |  {pct:>5.1f}%")

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#0891B2', '#EA580C', '#DC2626', '#334155', '#22D3EE', '#94A3B8'][:BEST_K]
bars = ax.bar(cluster_counts.index, cluster_counts.values, color=colors, edgecolor="white")

ax.set_title(f"Cluster Size Distribution (k={BEST_K})", fontsize=13, fontweight="bold")
ax.set_xlabel("Cluster ID", fontsize=11)
ax.set_ylabel("Number of Reviews", fontsize=11)

# Annotate each bar with count and percentage
for bar, (cluster_id, count) in zip(bars, cluster_counts.items()):
    pct = 100 * count / len(df_merged)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + len(df_merged) * 0.01,
        f"{count:,}\n({pct:.1f}%)",
        ha="center", va="bottom", fontsize=9
    )

plt.tight_layout()
cluster_size_path = os.path.join(PLOTS_DIR, "nb04_cluster_sizes.png")
plt.savefig(cluster_size_path, dpi=150, bbox_inches="tight")
print(f"\n✓ Cluster size plot saved → {cluster_size_path}")
plt.show()

---
## Section 4 — Dimensionality Reduction & Visualisation

The embeddings are 384-dimensional — impossible to visualise directly.
I use **UMAP** (Uniform Manifold Approximation and Projection) to project them down to 2D
while preserving both local and global structure.

**Why UMAP over t-SNE?**
- UMAP preserves **global structure** better — clusters that are far apart in high-D stay far apart in 2D
- UMAP is **faster** — O(n log n) vs t-SNE's O(n²)
- UMAP has **meaningful distances** — points that are close in the projection are genuinely similar

I work with a sample of ~5 000 reviews for the UMAP projection since it's still computationally heavy.

In [ ]:
# ── 4.1  Sample for UMAP visualisation ──────────────────────────────────────
# UMAP complexity scales with n·log(n), but visualisation with >10k points
# gets slow and the scatter plot becomes unreadable anyway.
#
# I take a stratified sample — proportional representation from each cluster.
# Expected output: sample indices, sample size printed.

UMAP_SAMPLE_SIZE = min(5_000, len(df_merged))  # Cap at 5k for speed

print(f"Sampling {UMAP_SAMPLE_SIZE:,} reviews for UMAP visualisation ...")

# Stratified by cluster to preserve the cluster proportions
umap_sample_indices = []
for cluster_id in range(BEST_K):
    cluster_mask = df_merged["cluster"] == cluster_id
    cluster_df = df_merged[cluster_mask]

    # How many samples from this cluster?
for cluster_id in range(BEST_K):
    cluster_mask = df_merged["cluster"] == cluster_id
    cluster_df = df_merged[cluster_mask]

    # Proportional allocation — floor to avoid overshoot
    n_proportional = int(UMAP_SAMPLE_SIZE * len(cluster_df) / len(df_merged))
    n_samples = max(1, n_proportional)

    sampled = cluster_df.sample(n=n_samples, random_state=SEED)
    umap_sample_indices.extend(sampled.index.tolist())


# Extract the corresponding embeddings and cluster labels
umap_embeddings = embeddings[umap_sample_indices]
umap_clusters = df_merged.loc[umap_sample_indices, "cluster"].values

print(f"  Sample size : {len(umap_sample_indices):,} reviews")
print(f"  Embeddings  : {umap_embeddings.shape}")

In [ ]:
# ── 4.2  Fit UMAP and project to 2D ─────────────────────────────────────────
# UMAP parameters tuned for visualisation:
#   - n_neighbors=15: balance between local detail (low) and global structure (high)
#   - min_dist=0.1: how tightly points are packed — lower = tighter clusters
#   - metric='cosine': uses cosine distance, which works well with normalised embeddings
#
# Expected output: 2D coordinates array of shape (n_sample, 2).

print("Fitting UMAP for 2D projection ...")

umap_reducer = UMAP(
    n_neighbors=15,         # Local vs global structure balance
    min_dist=0.1,           # Minimum distance between points in the projection
    n_components=2,         # Project to 2D for scatter plot
    metric="cosine",        # Cosine distance (suits normalised embeddings)
    random_state=SEED,
    verbose=True,           # Show progress bar
)

umap_coords = umap_reducer.fit_transform(umap_embeddings)  # Shape: (n, 2)

print(f"\n✓ UMAP projection complete.")
print(f"  Output shape: {umap_coords.shape}")
print(f"  X range     : [{umap_coords[:,0].min():.2f}, {umap_coords[:,0].max():.2f}]")
print(f"  Y range     : [{umap_coords[:,1].min():.2f}, {umap_coords[:,1].max():.2f}]")

In [ ]:
# ── 4.3  Plot UMAP scatter coloured by cluster ──────────────────────────────
# This is the key visual output — it shows whether the clustering found
# meaningful separations in the embedding space.
# Well-separated coloured groups = good clustering.
#
# Expected output: UMAP scatter plot saved as PNG.

fig, ax = plt.subplots(figsize=(10, 8))

# Colour palette — one distinct colour per cluster
cluster_palette = ['#0891B2', '#EA580C', '#DC2626', '#334155', '#22D3EE', '#94A3B8'][:BEST_K]

for cluster_id in range(BEST_K):
    mask = umap_clusters == cluster_id
    ax.scatter(
        umap_coords[mask, 0],
        umap_coords[mask, 1],
        c=[cluster_palette[cluster_id]],
        label=f"Cluster {cluster_id}",
        s=5,               # Small marker size — many points
        alpha=0.6,         # Semi-transparent to show density
        edgecolors="none", # No border — cleaner with many points
    )

ax.set_title(
    f"UMAP Projection of MiniLM Embeddings — Coloured by Cluster (k={BEST_K})",
    fontsize=14, fontweight="bold"
)
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(
    markerscale=4,          # Make legend markers bigger
    fontsize=10,
    title="Cluster",
    title_fontsize=11,
    loc="upper right",
)
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
umap_path = os.path.join(PLOTS_DIR, "nb04_umap_clusters.png")
plt.savefig(umap_path, dpi=150, bbox_inches="tight")
print(f"✓ UMAP cluster plot saved → {umap_path}")
plt.show()

In [ ]:
# ── 4.4  Plot UMAP scatter coloured by sentiment ────────────────────────────
# Overlaying the sentiment colours on the same UMAP projection reveals
# whether clusters correspond to sentiment patterns.
# For example, a cluster might contain mostly positive or negative reviews.
#
# I use the 'predicted_label' from NB02/NB03 if available, otherwise 'label' (true sentiment).
# Expected output: second UMAP scatter plot saved as PNG.

if df_preds is not None and "predicted_label" in df_merged.columns:
    sentiment_col = "predicted_label"
    sentiment_title = "Predicted Sentiment"
else:
    # Use true label from NB01 (mapped to names)
    df_merged["sentiment_name"] = df_merged["label"].map(label_names)
    sentiment_col = "sentiment_name"
    sentiment_title = "True Sentiment (Label)"

umap_sentiments = df_merged.loc[umap_sample_indices, sentiment_col].values

fig, ax = plt.subplots(figsize=(10, 8))

sentiment_palette = {"Negative": "#DC2626", "Neutral": "#EA580C", "Positive": "#0891B2"}

for sentiment_name, color in sentiment_palette.items():
    mask = umap_sentiments == sentiment_name
    n_points = mask.sum()
    if n_points == 0:
        continue
    ax.scatter(
        umap_coords[mask, 0],
        umap_coords[mask, 1],
        c=[color],
        label=f"{sentiment_name} ({n_points:,})",
        s=5,
        alpha=0.6,
        edgecolors="none",
    )

ax.set_title(
    f"UMAP Projection — Coloured by {sentiment_title}",
    fontsize=14, fontweight="bold"
)
ax.set_xlabel("UMAP Dimension 1", fontsize=11)
ax.set_ylabel("UMAP Dimension 2", fontsize=11)
ax.legend(markerscale=4, fontsize=10, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
umap_sent_path = os.path.join(PLOTS_DIR, "nb04_umap_sentiment.png")
plt.savefig(umap_sent_path, dpi=150, bbox_inches="tight")
print(f"✓ UMAP sentiment plot saved → {umap_sent_path}")
plt.show()

---
## Section 5 — Cluster Analysis

Now I analyse what each cluster actually **means**. A clustering algorithm only assigns numbers;
it is up to me to interpret those numbers as product categories.

I do this in three ways:
1. **Top TF-IDF terms per cluster** — what words uniquely characterise each cluster?
2. **Sentiment distribution per cluster** — are some clusters more positive/negative than others?
3. **Rating statistics per cluster** — what is the average star rating in each cluster?

Together, these let me assign human-readable labels like "Electronics — Mostly Positive"
or "Books — Mixed Sentiment".

In [ ]:
# ── 5.1  Top TF-IDF terms per cluster ────────────────────────────────────────
# I fit a TF-IDF vectoriser on the entire corpus, then for each cluster
# I aggregate the TF-IDF scores to find which terms are most distinctive.
#
# Approach: for each cluster, sum the TF-IDF scores across all its reviews
# and take the top N terms with highest total score.
#
# Expected output: top 10 terms printed per cluster.

print("Computing TF-IDF terms per cluster ...")

# ── Fit TF-IDF vectoriser on all texts ──
# max_features=5000 keeps the vocabulary manageable in RAM
# stop_words='english' removes common words like 'the', 'and', 'is'
# ngram_range=(1,2) includes both unigrams and bigrams (e.g., 'battery life')
tfidf = TfidfVectorizer(
    max_features=5_000,
    stop_words="english",
    ngram_range=(1, 2),          # Unigrams + bigrams for richer terms
    min_df=5,                    # Ignore terms appearing in fewer than 5 docs
    max_df=0.7,                  # Ignore terms appearing in >70% of docs (too common)
)

tfidf_matrix = tfidf.fit_transform(df_merged["text"])  # Shape: (n_reviews, n_features)
feature_names = tfidf.get_feature_names_out()            # List of term strings

print(f"TF-IDF vocabulary size: {len(feature_names):,} terms")
print(f"TF-IDF matrix shape   : {tfidf_matrix.shape}")
print()

# ── Compute top terms per cluster ──
TOP_N_TERMS = 10  # How many top terms to show per cluster

for cluster_id in range(BEST_K):
    cluster_mask = df_merged["cluster"] == cluster_id
    cluster_indices = df_merged[cluster_mask].index  # Row indices in the full DataFrame

    # Convert positional indices to TF-IDF matrix row indices
    # (The order of rows in df_merged matches tfidf_matrix)
    cluster_tfidf = tfidf_matrix[cluster_indices]  # Sparse sub-matrix for this cluster

    # Sum TF-IDF scores across all reviews in this cluster (per term)
    # .sum(axis=0) returns a 1×V matrix; .A1 flattens it to a 1D array
    term_scores = cluster_tfidf.sum(axis=0).A1  # Shape: (n_features,)

    # Get indices of top-scoring terms (descending order)
    top_indices = np.argsort(term_scores)[::-1][:TOP_N_TERMS]

    # Print top terms with their scores
    n_in_cluster = cluster_mask.sum()
    print(f"Cluster {cluster_id} ({n_in_cluster:,} reviews):")
    for rank, idx in enumerate(top_indices, 1):
        term = feature_names[idx]
        score = term_scores[idx]
        print(f"  {rank:2d}. {term:<30s}  (TF-IDF sum: {score:.2f})")
    print()

In [ ]:
# ── 5.2  Sentiment distribution per cluster ──────────────────────────────────
# I compute the proportion of each sentiment class within each cluster.
# This reveals whether a cluster is mostly positive, negative, or mixed.
#
# Expected output: stacked bar chart of sentiment proportions per cluster.

# ── Use predicted labels if available, otherwise true labels ──
if df_preds is not None and "predicted_label" in df_merged.columns:
    sentiment_col = "predicted_label"
else:
    df_merged["sentiment_name"] = df_merged["label"].map(label_names)
    sentiment_col = "sentiment_name"

# ── Compute crosstab: cluster × sentiment ──
sentiment_crosstab = pd.crosstab(
    df_merged["cluster"],
    df_merged[sentiment_col],
    normalize="index"  # Normalise by row → proportions per cluster
) * 100  # Convert to percentages

print("Sentiment distribution per cluster (%):")
print(sentiment_crosstab.round(1).to_string())

# ── Stacked bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))

sentiment_colors = ["#DC2626", "#EA580C", "#0891B2"]  # Red, amber, green

sentiment_crosstab.plot(
    kind="bar",
    stacked=True,
    color=sentiment_colors,
    edgecolor="white",
    linewidth=0.8,
    ax=ax,
    legend=True,
)

ax.set_title(
    f"Sentiment Distribution per Cluster (k={BEST_K})",
    fontsize=14, fontweight="bold"
)
ax.set_xlabel("Cluster ID", fontsize=11)
ax.set_ylabel("Percentage of Reviews (%)", fontsize=11)
ax.set_ylim(0, 105)
ax.legend(title="Sentiment", fontsize=10, title_fontsize=11, loc="upper right")

# Add percentage labels on each segment
for c_idx, container in enumerate(ax.containers):
    labels = [f"{v:.0f}%" if v > 5 else "" for v in sentiment_crosstab.iloc[:, c_idx]]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8, color="white", fontweight="bold")

plt.tight_layout()
sentiment_cluster_path = os.path.join(PLOTS_DIR, "nb04_sentiment_per_cluster.png")
plt.savefig(sentiment_cluster_path, dpi=150, bbox_inches="tight")
print(f"\n✓ Sentiment-per-cluster plot saved → {sentiment_cluster_path}")
plt.show()

In [ ]:
# ── 5.3  Average rating per cluster ──────────────────────────────────────────
# The original rating (1–5 stars) from NB01 is a direct signal of product satisfaction.
# High average rating → cluster contains well-liked products.
# Low average rating → cluster contains problematic product categories.
#
# Expected output: table and bar chart of mean rating per cluster.

if "rating" in df_merged.columns:
    rating_stats = df_merged.groupby("cluster")["rating"].agg(["mean", "std", "count"])
    rating_stats["mean"] = rating_stats["mean"].round(2)
    rating_stats["std"] = rating_stats["std"].round(2)

    print("Rating statistics per cluster:")
    print(rating_stats.to_string())

    # ── Bar chart of mean rating ──
    fig, ax = plt.subplots(figsize=(8, 5))

    bars = ax.bar(
        rating_stats.index,
        rating_stats["mean"],
        yerr=rating_stats["std"],     # Error bars = std deviation
        capsize=8,                     # Width of error bar caps
        color=cluster_palette,
        edgecolor="white",
        linewidth=0.8,
    )

    ax.set_title(f"Average Star Rating per Cluster (k={BEST_K})", fontsize=14, fontweight="bold")
    ax.set_xlabel("Cluster ID", fontsize=11)
    ax.set_ylabel("Mean Rating (1–5 stars)", fontsize=11)
    ax.set_ylim(0, 5.5)
    ax.axhline(y=3.0, color="#334155", linestyle="--", linewidth=1, alpha=0.7, label="Neutral (3★)")
    ax.legend(fontsize=10)

    # Annotate each bar with the mean value
    for bar, mean_val in zip(bars, rating_stats["mean"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            f"{mean_val:.2f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold"
        )

    plt.tight_layout()
    rating_path = os.path.join(PLOTS_DIR, "nb04_avg_rating_per_cluster.png")
    plt.savefig(rating_path, dpi=150, bbox_inches="tight")
    print(f"\n✓ Rating-per-cluster plot saved → {rating_path}")
    plt.show()
else:
    print("⚠️  No 'rating' column found — skip rating analysis.")

In [ ]:
# ── 5.4  Cluster profile summary table ───────────────────────────────────────
# I consolidate all cluster information into a single summary table:
# size, dominant sentiment, average rating, and top 3 terms.
# This is the "cheat sheet" for understanding what each cluster represents.
#
# Expected output: formatted summary table printed.

print("╔══════════════════════════════════════════════════════════════════════════════╗")
print("║                         CLUSTER PROFILE SUMMARY                               ║")
print("╠══════════════════════════════════════════════════════════════════════════════╣")

for cluster_id in range(BEST_K):
    cluster_mask = df_merged["cluster"] == cluster_id
    n_reviews = cluster_mask.sum()
    pct_total = 100 * n_reviews / len(df_merged)

    # Dominant sentiment
    if sentiment_col in df_merged.columns:
        dominant_sent = df_merged.loc[cluster_mask, sentiment_col].mode()
        dominant_sent = dominant_sent.iloc[0] if len(dominant_sent) > 0 else "N/A"
    else:
        dominant_sent = "N/A"

    # Average rating
    if "rating" in df_merged.columns:
        avg_rating = df_merged.loc[cluster_mask, "rating"].mean()
        rating_str = f"{avg_rating:.2f} ★"
    else:
        rating_str = "N/A"

    # Top 3 terms (already computed in 5.1 — reuse the same logic)
    cluster_indices = df_merged[cluster_mask].index
    cluster_tfidf = tfidf_matrix[cluster_indices]
    term_scores = cluster_tfidf.sum(axis=0).A1
    top3_indices = np.argsort(term_scores)[::-1][:3]
    top3_terms = ", ".join(feature_names[i] for i in top3_indices)

    print(f"║  Cluster {cluster_id}:")
    print(f"║    Size              : {n_reviews:,} reviews ({pct_total:.1f}%)")
    print(f"║    Dominant sentiment: {dominant_sent}")
    print(f"║    Avg rating        : {rating_str}")
    print(f"║    Top terms         : {top3_terms}")
    print(f"║")

print("╚══════════════════════════════════════════════════════════════════════════════╝")

---
## Section 6 — Export Results

I save all outputs so Notebook 05 (Review Summarisation) can use them without re-running
the clustering pipeline.

**Outputs**:
- `clusters.csv` — review text + cluster assignments + sentiment (for NB05 summarisation)
- `cluster_centroids.npy` — KMeans centroids (for assigning new reviews to clusters)
- `cluster_profiles.json` — structured summary of each cluster's characteristics

In [ ]:
# ── 6.1  Save cluster assignments to CSV ────────────────────────────────────
# This CSV is the primary output consumed by NB05 (review summarisation).
# It includes the review text, cluster ID, and sentiment predictions.
#
# Expected output: CSV file saved to OUTPUT_DIR.

# ── Select columns for export ──
export_cols = ["text", "cluster"]

if "rating" in df_merged.columns:
    export_cols.append("rating")
if df_preds is not None and "predicted_label" in df_merged.columns:
    export_cols.append("predicted_label")
    export_cols.append("confidence")
if "label" in df_merged.columns:
    export_cols.append("label")  # True label for reference

# ── Build export DataFrame ──
df_export = df_merged[export_cols].copy()

# ── Save to CSV ──
clusters_csv_path = os.path.join(OUTPUT_DIR, "clusters.csv")
df_export.to_csv(
    clusters_csv_path,
    index=False,          # Do not write the integer row index as a column
    encoding="utf-8",     # UTF-8 to preserve special characters in review text
)

print(f"✓ Cluster assignments saved → {clusters_csv_path}")
print(f"  Rows       : {len(df_export):,}")
print(f"  Columns    : {list(df_export.columns)}")
print(f"  File size  : {os.path.getsize(clusters_csv_path) / 1e6:.2f} MB")

In [ ]:
# ── 6.2  Save KMeans centroids ──────────────────────────────────────────────
# The centroids are the "prototype" embedding for each cluster.
# NB05 or a future web app can use them to assign new reviews to clusters
# by finding the nearest centroid (no re-training needed).
#
# Expected output: .npy file saved to MODELS_DIR.

centroids = final_kmeans.cluster_centers_  # Shape: (BEST_K, 384)

centroids_path = os.path.join(MODELS_DIR, "cluster_centroids.npy")
np.save(centroids_path, centroids)

print(f"✓ Cluster centroids saved → {centroids_path}")
print(f"  Shape      : {centroids.shape}")
print(f"  File size  : {os.path.getsize(centroids_path) / 1e3:.2f} KB")

In [ ]:
# ── 6.3  Save cluster profiles as JSON ──────────────────────────────────────
# Structured summary of each cluster's key characteristics.
# Useful for the final report and for NB05 to generate targeted summaries.
#
# Expected output: JSON file saved to OUTPUT_DIR.

cluster_profiles = {
    "model_embedding": EMBEDDING_MODEL_NAME,
    "model_clustering": "MiniBatchKMeans",
    "n_clusters": BEST_K,
    "silhouette_score": float(FINAL_SIL),
    "total_reviews": len(df_merged),
    "clusters": []
}

for cluster_id in range(BEST_K):
    cluster_mask = df_merged["cluster"] == cluster_id
    n_reviews = cluster_mask.sum()

    # Top 5 terms
    cluster_indices = df_merged[cluster_mask].index
    cluster_tfidf = tfidf_matrix[cluster_indices]
    term_scores = cluster_tfidf.sum(axis=0).A1
    top5_indices = np.argsort(term_scores)[::-1][:5]
    top5_terms = [feature_names[i] for i in top5_indices]

    # Sentiment proportions
    if sentiment_col in df_merged.columns:
        sent_counts = df_merged.loc[cluster_mask, sentiment_col].value_counts().to_dict()
        sent_pct = {k: round(100 * v / n_reviews, 1) for k, v in sent_counts.items()}
    else:
        sent_pct = {}

    # Average rating
    avg_rating = float(df_merged.loc[cluster_mask, "rating"].mean()) if "rating" in df_merged.columns else None

    cluster_profiles["clusters"].append({
        "cluster_id": int(cluster_id),
        "size": int(n_reviews),
        "pct_of_total": round(100 * n_reviews / len(df_merged), 1),
        "top_terms": top5_terms,
        "sentiment_distribution_pct": sent_pct,
        "avg_rating": avg_rating,
    })

profiles_path = os.path.join(OUTPUT_DIR, "cluster_profiles.json")
with open(profiles_path, "w") as f:
    json.dump(cluster_profiles, f, indent=2)

print(f"✓ Cluster profiles saved → {profiles_path}")
print(f"  File size: {os.path.getsize(profiles_path):,} bytes")

In [ ]:
# ── 6.4  Output file checklist ───────────────────────────────────────────────
# Verify that all expected output files were created before wrapping up.

expected_outputs = {
    "Cluster assignments CSV"  : os.path.join(OUTPUT_DIR, "clusters.csv"),
    "Cluster profiles JSON"    : os.path.join(OUTPUT_DIR, "cluster_profiles.json"),
    "Cluster centroids NPY"    : os.path.join(MODELS_DIR, "cluster_centroids.npy"),
    "Embeddings NPZ"           : os.path.join(OUTPUT_DIR, "embeddings_minilm.npz"),
    "K-selection plot"         : os.path.join(PLOTS_DIR,  "nb04_k_selection.png"),
    "Cluster sizes plot"       : os.path.join(PLOTS_DIR,  "nb04_cluster_sizes.png"),
    "UMAP clusters plot"       : os.path.join(PLOTS_DIR,  "nb04_umap_clusters.png"),
    "UMAP sentiment plot"      : os.path.join(PLOTS_DIR,  "nb04_umap_sentiment.png"),
    "Sentiment per cluster"    : os.path.join(PLOTS_DIR,  "nb04_sentiment_per_cluster.png"),
    "Avg rating per cluster"   : os.path.join(PLOTS_DIR,  "nb04_avg_rating_per_cluster.png"),
}

print("Output file checklist:")
print("-" * 55)
all_ok = True
for label, path in expected_outputs.items():
    exists = os.path.exists(path)
    status = "✓  OK" if exists else "✗  MISSING"
    if not exists:
        all_ok = False
    print(f"  {label:30s}  {status}")

print("-" * 55)
if all_ok:
    print("All output files present — Notebook 04 complete!")
else:
    print("WARNING: Some output files are missing. Check the cells above for errors.")

---
## Notebook 04 Complete ✓

**Summary of work done in this notebook:**

- Loaded the Arrow-format test split produced by Notebook 01
- Merged sentiment predictions from Notebook 02 (DistilBERT) and/or Notebook 03 (RoBERTa)
- Encoded all review texts into 384-dimensional embeddings with `all-MiniLM-L6-v2`
- Applied `MiniBatchKMeans` to discover product categories — swept k=2…10
- Evaluated clustering quality with **Silhouette Score** and **Elbow Method**
- Selected the optimal number of clusters (k) in the 4–6 target range
- Visualised clusters in 2D with **UMAP** (coloured by cluster and by sentiment)
- Extracted **top TF-IDF terms** per cluster to interpret what each cluster represents
- Analysed **sentiment distribution** and **average rating** per cluster
- Exported cluster assignments (`clusters.csv`), centroids (`.npy`), and profiles (`.json`)

**Next step:** Notebook 05 — Review Summarisation (uses `clusters.csv` to generate blog-style articles)

In [ ]:
# ── Final Summary Print ───────────────────────────────────────────────────────
# I print a clean end-of-notebook summary so anyone opening the notebook
# can immediately see what was produced without scrolling through all cells.
#
# Expected output: formatted summary block.

print("=" * 60)
print("  NOTEBOOK 04 — COMPLETE")
print("  Product Category Clustering")
print("=" * 60)
print()
print(f"  Embedding model  : {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIM}-dim)")
print(f"  Clustering algo  : MiniBatchKMeans (k={BEST_K})")
print(f"  Reviews clustered: {len(df_merged):,}")
print(f"  Silhouette Score : {FINAL_SIL:.4f}")
print()
print(f"  Clusters found (k={BEST_K}):")
for cluster_id in range(BEST_K):
    n_reviews = (df_merged["cluster"] == cluster_id).sum()
    pct = 100 * n_reviews / len(df_merged)
    print(f"    Cluster {cluster_id}: {n_reviews:,} reviews ({pct:.1f}%)")
print()
print(f"  Outputs saved:")
print(f"    {os.path.join(OUTPUT_DIR, 'clusters.csv')}")
print(f"    {os.path.join(OUTPUT_DIR, 'cluster_profiles.json')}")
print(f"    {os.path.join(MODELS_DIR, 'cluster_centroids.npy')}")
print(f"    {os.path.join(OUTPUT_DIR, 'embeddings_minilm.npz')}")
print(f"    {PLOTS_DIR}/  (6 PNG visualisations)")
print()
print("  ➡️  Next: Notebook 05 — Review Summarisation")
print("=" * 60)